## Crawling Naver Stock Repots

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
# 1. URL
url = 'https://finance.naver.com/research/company_list.naver?page=1'

In [4]:
# 2. request(URL) > response(HTML)
response = requests.get(url)
response

<Response [200]>

In [5]:
response.text[:200]

'<!--  global include -->\n\n\t\n\t\n\t\n\t\n\t\n<html lang=\'ko\'>\n<head>\n\n\n\t\n\t\n\t\t\n\t\t\t\n\t\t\t\n\t\t\t\t<title>종목분석 리포트 : 네이버페이 증권</title>\n\t\t\t\n\t\t\n\t\n\n\n\n\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n\n'

In [6]:
# 3. HTML > BeautifulSoup > css-selector > DataFrame

In [7]:
dom = BeautifulSoup(response.content, 'html.parser')
type(dom) # select(css-selector), select_one()

bs4.BeautifulSoup

In [10]:
selector = 'table.type_1 > tr'
elements = dom.select(selector)
len(elements)

49

In [14]:
elements[1]

<tr><td class="blank_07" colspan="6"></td></tr>

In [15]:
elements[2]

<tr>
<td style="padding-left:10">
<a class="stock_item" href="/item/main.naver?code=066570" title="LG전자">LG전자</a>
</td>
<td><a href="company_read.naver?nid=77126&amp;page=1">AI데이터센터 냉각 시장 공략</a><img alt="NEW" class="ico_new" height="8" src="https://ssl.pstatic.net/imgstock/images5/ico_research_new.gif" width="8"/></td>
<td>교보증권</td>
<td class="file"><a href="https://stock.pstatic.net/stock-research/company/34/20240923_company_481199000.pdf" target="_blank"><img align="absmiddle" alt="pdf" src="https://ssl.pstatic.net/imgstock/images5/down.gif"/></a></td>
<td class="date" style="padding-left:5px">24.09.23</td>
<td class="date">402</td>
</tr>

In [16]:
element = elements[2]
tag = element.select('td')
len(tag), tag

(6,
 [<td style="padding-left:10">
  <a class="stock_item" href="/item/main.naver?code=066570" title="LG전자">LG전자</a>
  </td>,
  <td><a href="company_read.naver?nid=77126&amp;page=1">AI데이터센터 냉각 시장 공략</a><img alt="NEW" class="ico_new" height="8" src="https://ssl.pstatic.net/imgstock/images5/ico_research_new.gif" width="8"/></td>,
  <td>교보증권</td>,
  <td class="file"><a href="https://stock.pstatic.net/stock-research/company/34/20240923_company_481199000.pdf" target="_blank"><img align="absmiddle" alt="pdf" src="https://ssl.pstatic.net/imgstock/images5/down.gif"/></a></td>,
  <td class="date" style="padding-left:5px">24.09.23</td>,
  <td class="date">402</td>])

In [18]:
data = {}
data['stock_name'] = tag[0].select_one('a').text
data['stock_link'] = tag[0].select_one('a').get('href')
data['title'] = tag[1].select_one('a').text
data['title_link'] = tag[1].select_one('a').get('href')
data['writer'] = tag[2].text
data['pdf_link'] = tag[3].select_one('a').get('href')
data['date'] = tag[4].text
data['pv'] = tag[5].text
data

{'stock_name': 'LG전자',
 'stock_link': '/item/main.naver?code=066570',
 'title': 'AI데이터센터 냉각 시장 공략',
 'title_link': 'company_read.naver?nid=77126&page=1',
 'writer': '교보증권',
 'pdf_link': 'https://stock.pstatic.net/stock-research/company/34/20240923_company_481199000.pdf',
 'date': '24.09.23',
 'pv': '402'}

In [47]:
rows = []
for idx, element in enumerate(elements):
    tag = element.select('td')
    print(idx, len(tag))
    if len(tag) == 6:
        data = {}
        data['stock_name'] = tag[0].select_one('a').text
        data['stock_link'] = tag[0].select_one('a').get('href')
        data['title'] = tag[1].select_one('a').text
        data['title_link'] = tag[1].select_one('a').get('href')
        data['writer'] = tag[2].text
        data['pdf_link'] = tag[3].select_one('a').get('href')
        data['date'] = tag[4].text
        data['pv'] = tag[5].text
        rows.append(data)

0 0
1 1
2 6
3 6
4 6
5 6
6 6
7 1
8 1
9 1
10 6
11 6
12 6
13 6
14 6
15 1
16 1
17 1
18 6
19 6
20 6
21 6
22 6
23 1
24 1
25 1
26 6
27 6
28 6
29 6
30 6
31 1
32 1
33 1
34 6
35 6
36 6
37 6
38 6
39 1
40 1
41 1
42 6
43 6
44 6
45 6
46 6
47 1
48 1


In [45]:
df = pd.DataFrame(rows)
df

,stock_name,stock_link,title,title_link,writer,pdf_link,date,pv
0,LG전자,/item/main.naver?code=066570,AI데이터센터 냉각 시장 공략,company_read.naver?nid=77126&page=1,교보증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,402
1,케이엔솔,/item/main.naver?code=053080,설계와 시공능력으로 액침냉각 사업 확대,company_read.naver?nid=77125&page=1,교보증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,190
2,GST,/item/main.naver?code=083450,"액침냉각, 기술적 강점을 확보해 나가는 중",company_read.naver?nid=77124&page=1,교보증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,228
3,코스맥스,/item/main.naver?code=192820,3Q24 Preview: 국내 수주 강세 VS 중국 부진 ..,company_read.naver?nid=77123&page=1,하나증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,135
4,유한양행,/item/main.naver?code=000100,Re-rating 구간 돌입,company_read.naver?nid=77122&page=1,유진투자증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,704
5,LIG넥스원,/item/main.naver?code=079550,높아지는 Peak sales,company_read.naver?nid=77121&page=1,미래에셋증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,347
6,디앤디파마텍,/item/main.naver?code=347850,"Metsera, 너는 계획이 다 있구나",company_read.naver?nid=77120&page=1,키움증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,297
7,디지털대성,/item/main.naver?code=068930,"의대 열풍, 나만 믿어",company_read.naver?nid=77119&page=1,신한투자증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,246
8,휠라홀딩스,/item/main.naver?code=081660,"속도가 느려도, 방향성은 맞다",company_read.naver?nid=77118&page=1,키움증권,https://stock.pstatic.net/stock-research/compa...,24.09.23,234
9,SK바이오팜,/item/main.naver?code=326030,"빅파마들의 RPT 방향, 우리도 간다",company_read.naver?nid=77117&page=1,미래에셋증권,https://stock.pstatic.net/stock-research/compa...,24.09.20,3047


In [ ]:
# File Download

In [49]:
# os package : 파일 시스템을 관리하는 파이썬 패키지
import os
os.listdir()

['.ipynb_checkpoints',
 '00_python.ipynb',
 '01_requests_naver_stock.ipynb',
 '02_requests_daum_exchange.ipynb',
 '03_rest_api.ipynb',
 '04_requests_zigbang.ipynb',
 '05_html.ipynb',
 '06_css_selector.ipynb',
 '07_naver_relational_keywords.ipynb',
 '08_gmarket.ipynb',
 '08_naver_stock_report.ipynb',
 '09_selenium.ipynb',
 '10_xpath.ipynb',
 '11_iterator_generator.ipynb',
 '12_scrapy.ipynb']

In [90]:
if not os.path.exists(path):
    os.makedirs(path)

In [63]:
path = 'reports'
# 디렉토리, 파일 존재 여부 확인
os.path.exists(path)

True

In [65]:
os.listdir()

['.ipynb_checkpoints',
 '00_python.ipynb',
 '01_requests_naver_stock.ipynb',
 '02_requests_daum_exchange.ipynb',
 '03_rest_api.ipynb',
 '04_requests_zigbang.ipynb',
 '05_html.ipynb',
 '06_css_selector.ipynb',
 '07_naver_relational_keywords.ipynb',
 '08_gmarket.ipynb',
 '08_naver_stock_report.ipynb',
 '09_selenium.ipynb',
 '10_xpath.ipynb',
 '11_iterator_generator.ipynb',
 '12_scrapy.ipynb',
 'reports']

In [71]:
# 1. url
title = df.loc[0, 'title']
pdf_link = df.loc[0, 'pdf_link']
title, pdf_link

('AI데이터센터 냉각 시장 공략',
 'https://stock.pstatic.net/stock-research/company/34/20240923_company_481199000.pdf')

In [77]:
# 2. request(url) -> response(pdf)
response = requests.get(pdf_link)
response

<Response [200]>

In [81]:
# 3. pdf save
filename = f'{path}/{title}.pdf'
with open(filename, 'wb') as file:
    file.write(response.content)

In [84]:
os.listdir('reports')

['.ipynb_checkpoints', 'AI데이터센터 냉각 시장 공략.pdf']

In [107]:
import shutil
shutil.rmtree(path) # 디렉토리 제거
os.makedirs(path)

In [109]:
os.listdir(path)

[]

In [111]:
for idx, row in df.iterrows():
    title, pdf_link = row['title'], row['pdf_link']
    if ':' in title: 
        title = title.replace(':', ' -')
    response = requests.get(pdf_link)
    filename = f'{path}/{title}.pdf'
    with open(filename, 'wb') as file:
        file.write(response.content)

In [112]:
os.listdir(path)

['2024년 3분기 부진 터널 통과해야!.pdf',
 '3Q24 Preview - 국내 수주 강세 VS 중국 부진 ...pdf',
 '3Q24 일시적인 실적 둔화,  분위기 반전이 필...pdf',
 '3분기는 비수기, 그러나   주주 가치 환원에 ...pdf',
 'AI데이터센터 냉각 시장 공략.pdf',
 'Metsera, 너는 계획이 다 있구나.pdf',
 'Moment of Truth.pdf',
 'Re-rating 구간 돌입.pdf',
 '가치가 높아지는 F박스와 K패션 해외 진출 수...pdf',
 '경쟁력 있는 CDMO, 여기에도 있다.pdf',
 '경쟁사 스트리머 이적, 트래픽 유입 기대.pdf',
 '경증 아토피 치료의 글로벌  시장 판도를 바...pdf',
 '금리 하락으로 미국에서 훈풍이 불어온다.pdf',
 '길어지고 있는 기다림.pdf',
 '높아지는 Peak sales.pdf',
 '롯데렌탈의 쏘카 지분 추가취득 당분간 중단.pdf',
 '미국 타워 판가도 인상.pdf',
 '빅파마들의 RPT 방향, 우리도 간다.pdf',
 '설계와 시공능력으로 액침냉각 사업 확대.pdf',
 '속도가 느려도, 방향성은 맞다.pdf',
 '액침냉각, 기술적 강점을 확보해 나가는 중.pdf',
 '엔지니어링 플라스틱 소재 개발 및 고도화로 ...pdf',
 '엘사.pdf',
 '의대 열풍, 나만 믿어.pdf',
 '이튼의 견고한 성장 파트너.pdf',
 '지속가능한 성장.pdf',
 '코스닥 방사성의약품 기업으로 진입 .pdf',
 '콥데이 후기 - 8.6G OLED 투자 본격화 수혜 기...pdf',
 '확대되는 TROP2 ADC 치료제 시장.pdf',
 '환율 모멘텀 발생 예상. 자사주 추가 매입 가...pdf']

In [ ]:
# tick(java) : pdf -> text